In [ ]:
# Install required packages (run once)
!pip install torch torchvision timm albumentations opencv-python scikit-learn matplotlib seaborn pillow tqdm wandb imblearn

# If you want to use Metal GPU acceleration on M4 Mac
!pip install --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cpu

In [ ]:
# Core imports
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from tqdm import tqdm
import random
import warnings
warnings.filterwarnings('ignore')

# PyTorch
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
from torchvision import models

# Timm for advanced models
import timm

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# ML utilities
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                            f1_score, confusion_matrix, classification_report, roc_auc_score)
from sklearn.model_selection import StratifiedKFold
from imblearn.over_sampling import SMOTE

# =====================
# 🔹 Reproducibility
# =====================
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    
# For deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# =====================
# 🔹 Device configuration (M4 Mac friendly)
# =====================
if torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple Metal GPU
    print("✅ Using MPS (Metal Performance Shaders) on Apple Silicon")
elif torch.cuda.is_available():
    device = torch.device("cuda")
    print("✅ Using CUDA (NVIDIA GPU)")
else:
    device = torch.device("cpu")
    print("⚠️ Using CPU only (slow)")

print(f"PyTorch version: {torch.__version__}")
print(f"Active device: {device}")


In [ ]:
#IMPORTING DATA

In [ ]:
import zipfile
import os

# Update this path
zip_path = "/Users/mahendra/Downloads/archive.zip"
import os
from pathlib import Path

# Check what was extracted
extract_to = "./extracted_data"

def explore_extracted_data():
    print("Exploring extracted data structure:")
    print("=" * 40)

    if os.path.exists(extract_to):
        for root, dirs, files in os.walk(extract_to):
            level = root.replace(extract_to, '').count(os.sep)
            indent = ' ' * 2 * level
            print(f'{indent}{os.path.basename(root)}/')
            subindent = ' ' * 2 * (level + 1)
            for file in files[:5]:  # Show first 5 files only
                print(f'{subindent}{file}')
            if len(files) > 5:
                print(f'{subindent}... and {len(files) - 5} more files')
    else:
        print(f"Directory {extract_to} not found!")

explore_extracted_data()

# Simple extraction
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_to)

print("Extraction complete!")
print(f"Files extracted to: {extract_to}")

In [ ]:
#Configuration and Hyperparameters

In [ ]:
class Config:
    # Paths
    DATA_ROOT = 'extracted_data/AutismDataset'
    TRAIN_DIR = os.path.join(DATA_ROOT, 'train')
    VALID_DIR = os.path.join(DATA_ROOT, 'valid')
    TEST_DIR = os.path.join(DATA_ROOT, 'test')
    
    # Model paths
    CHECKPOINT_DIR = 'checkpoints'
    RESULTS_DIR = 'results'
    
    # Training parameters
    IMG_SIZE = 224
    BATCH_SIZE = 16  # Optimized for M4 Mac
    EPOCHS = 15
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    NUM_WORKERS = 0  # M4 Mac optimized
    
    # Model parameters
    DROPOUT_RATE = 0.3
    MC_DROPOUT_PASSES = 20
    RESNET_LAYERS_TO_FINETUNE = 20
    
    # Augmentation
    USE_AUTOAUGMENT = True
    
    # Training strategy
    USE_FOCAL_LOSS = True
    USE_SAM = False  # Set True if you install SAM optimizer
    LABEL_SMOOTHING = 0.1
    
    # Ensemble
    NUM_MODELS = 3
    TTA_TRANSFORMS = 5
    
    # Other
    SEED = 42
    NUM_CLASSES = 2

# Create directories
os.makedirs(Config.CHECKPOINT_DIR, exist_ok=True)
os.makedirs(Config.RESULTS_DIR, exist_ok=True)

config = Config()
print("Configuration loaded successfully!")

In [ ]:
#Data Exploration and Visualization

In [ ]:
def explore_dataset(data_root):
    """Explore the dataset structure and class distribution"""
    
    stats = {
        'train': {'Autistic': 0, 'Non_Autistic': 0},
        'valid': {'Autistic': 0, 'Non_Autistic': 0},
        'test': {'Autistic': 0, 'Non_Autistic': 0}
    }
    
    # Count train images (mixed format)
    train_files = os.listdir(config.TRAIN_DIR)
    for f in train_files:
        if 'Autistic' in f and 'Non_Autistic' not in f:
            stats['train']['Autistic'] += 1
        elif 'Non_Autistic' in f:
            stats['train']['Non_Autistic'] += 1
    
    # Count valid images (structured)
    for class_name in ['Autistic', 'Non_Autistic']:
        valid_path = os.path.join(config.VALID_DIR, class_name)
        if os.path.exists(valid_path):
            stats['valid'][class_name] = len(os.listdir(valid_path))
    
    # Count test images (mixed format)
    test_files = os.listdir(config.TEST_DIR)
    for f in test_files:
        if 'Autistic' in f and 'Non_Autistic' not in f:
            stats['test']['Autistic'] += 1
        elif 'Non_Autistic' in f:
            stats['test']['Non_Autistic'] += 1
    
    # Visualization
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    for idx, (split, counts) in enumerate(stats.items()):
        labels = list(counts.keys())
        values = list(counts.values())
        
        axes[idx].bar(labels, values, color=['#FF6B6B', '#4ECDC4'])
        axes[idx].set_title(f'{split.capitalize()} Split', fontsize=14, fontweight='bold')
        axes[idx].set_ylabel('Number of Images')
        
        for i, v in enumerate(values):
            axes[idx].text(i, v + 10, str(v), ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/dataset_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print statistics
    print("\n" + "="*50)
    print("DATASET STATISTICS")
    print("="*50)
    for split, counts in stats.items():
        total = sum(counts.values())
        print(f"\n{split.upper()}:")
        for class_name, count in counts.items():
            percentage = (count/total)*100 if total > 0 else 0
            print(f"  {class_name}: {count} ({percentage:.1f}%)")
        print(f"  Total: {total}")
    
    return stats

# Run exploration
dataset_stats = explore_dataset(config.DATA_ROOT)

In [ ]:
#Data Augmentation Pipeline

In [ ]:
class AutoAugmentTransforms:
    """Advanced augmentation pipeline with AutoAugment"""
    
    @staticmethod
    def get_train_transforms():
        return A.Compose([
            # Resize and basic transforms
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            
            # AutoAugment-style transforms
            A.OneOf([
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1.0),
                A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=1.0),
                A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1, p=1.0),
            ], p=0.8),
            
            A.OneOf([
                A.GaussianBlur(blur_limit=(3, 7), p=1.0),
                A.MedianBlur(blur_limit=5, p=1.0),
                A.MotionBlur(blur_limit=5, p=1.0),
            ], p=0.3),
            
            A.OneOf([
                A.GaussNoise(var_limit=(10.0, 50.0), p=1.0),
                A.ISONoise(color_shift=(0.01, 0.05), intensity=(0.1, 0.5), p=1.0),
            ], p=0.3),
            
            A.ShiftScaleRotate(
                shift_limit=0.1, 
                scale_limit=0.15, 
                rotate_limit=15, 
                border_mode=cv2.BORDER_CONSTANT,
                p=0.5
            ),
            
            A.OneOf([
                A.GridDistortion(num_steps=5, distort_limit=0.3, p=1.0),
                A.ElasticTransform(alpha=1, sigma=50, alpha_affine=50, p=1.0),
            ], p=0.3),
            
            A.CoarseDropout(
                max_holes=8, 
                max_height=32, 
                max_width=32, 
                min_holes=1,
                fill_value=0, 
                p=0.3
            ),
            
            # Normalization
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ])
    
    @staticmethod
    def get_valid_transforms():
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.Normalize(
                mean=[0.485, 0.456, 0.406],
                std=[0.229, 0.224, 0.225],
            ),
            ToTensorV2(),
        ])
    
    @staticmethod
    def get_tta_transforms():
        """Test-time augmentation transforms"""
        return [
            # Original
            A.Compose([
                A.Resize(config.IMG_SIZE, config.IMG_SIZE),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ]),
            # Horizontal flip
            A.Compose([
                A.Resize(config.IMG_SIZE, config.IMG_SIZE),
                A.HorizontalFlip(p=1.0),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ]),
            # Brightness
            A.Compose([
                A.Resize(config.IMG_SIZE, config.IMG_SIZE),
                A.RandomBrightnessContrast(brightness_limit=0.1, contrast_limit=0.1, p=1.0),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ]),
            # Slight rotation
            A.Compose([
                A.Resize(config.IMG_SIZE, config.IMG_SIZE),
                A.Rotate(limit=10, p=1.0),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ]),
            # Scale
            A.Compose([
                A.Resize(config.IMG_SIZE, config.IMG_SIZE),
                A.RandomScale(scale_limit=0.1, p=1.0),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2(),
            ]),
        ]

print("Augmentation pipeline created!")

# Visualize augmentations
def visualize_augmentations(image_path, num_examples=5):
    """Visualize augmentation effects"""
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    transform = AutoAugmentTransforms.get_train_transforms()
    
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    
    # Original
    axes[0].imshow(image)
    axes[0].set_title('Original', fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Augmented versions
    for idx in range(1, 6):
        augmented = transform(image=image)['image']
        # Denormalize for visualization
        aug_img = augmented.numpy().transpose(1, 2, 0)
        aug_img = aug_img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        aug_img = np.clip(aug_img, 0, 1)
        
        axes[idx].imshow(aug_img)
        axes[idx].set_title(f'Augmented {idx}', fontsize=12, fontweight='bold')
        axes[idx].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/augmentation_examples.png', dpi=300, bbox_inches='tight')
    plt.show()

# Find a sample image
sample_image = os.path.join(config.TRAIN_DIR, os.listdir(config.TRAIN_DIR)[0])
visualize_augmentations(sample_image)

In [ ]:
#Custom Dataset Class

In [ ]:
class AutismDataset(Dataset):
    """Custom dataset for Autism classification"""
    
    def __init__(self, root_dir, transform=None, mode='train'):
        self.root_dir = root_dir
        self.transform = transform
        self.mode = mode
        self.images = []
        self.labels = []
        
        self._load_data()
    
    def _load_data(self):
        """Load image paths and labels"""
        if self.mode in ['train', 'test']:
            # Mixed format: Autistic.X.jpg and Non_Autistic.X.jpg
            files = os.listdir(self.root_dir)
            for filename in files:
                if filename.endswith(('.jpg', '.jpeg', '.png')):
                    filepath = os.path.join(self.root_dir, filename)
                    
                    if 'Non_Autistic' in filename:
                        label = 0  # Non-Autistic
                    elif 'Autistic' in filename:
                        label = 1  # Autistic
                    else:
                        continue
                    
                    self.images.append(filepath)
                    self.labels.append(label)
        
        elif self.mode == 'valid':
            # Structured format: valid/Autistic/ and valid/Non_Autistic/
            for class_name, label in [('Non_Autistic', 0), ('Autistic', 1)]:
                class_dir = os.path.join(self.root_dir, class_name)
                if os.path.exists(class_dir):
                    for filename in os.listdir(class_dir):
                        if filename.endswith(('.jpg', '.jpeg', '.png')):
                            filepath = os.path.join(class_dir, filename)
                            self.images.append(filepath)
                            self.labels.append(label)
        
        print(f"{self.mode.capitalize()} set: {len(self.images)} images loaded")
        print(f"  Class distribution: Autistic={sum(self.labels)}, Non-Autistic={len(self.labels)-sum(self.labels)}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Apply transforms
        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']
        
        return image, label, img_path

# Create datasets
train_dataset = AutismDataset(
    root_dir=config.TRAIN_DIR,
    transform=AutoAugmentTransforms.get_train_transforms(),
    mode='train'
)

valid_dataset = AutismDataset(
    root_dir=config.VALID_DIR,
    transform=AutoAugmentTransforms.get_valid_transforms(),
    mode='valid'
)

test_dataset = AutismDataset(
    root_dir=config.TEST_DIR,
    transform=AutoAugmentTransforms.get_valid_transforms(),
    mode='test'
)

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=True,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=config.BATCH_SIZE,
    shuffle=False,
    num_workers=config.NUM_WORKERS,
    pin_memory=True
)

print("\nDataLoaders created successfully!")
print(f"Train batches: {len(train_loader)}")
print(f"Valid batches: {len(valid_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
#SMOTE for Class Imbalance (Feature-based)

In [ ]:
class FeatureExtractor(nn.Module):
    """Extract features for SMOTE application"""
    def __init__(self):
        super().__init__()
        # Use lightweight model for feature extraction
        self.model = models.resnet18(pretrained=True)
        self.model.fc = nn.Identity()  # Remove classification layer
    
    def forward(self, x):
        return self.model(x)

def extract_features_for_smote(dataloader, device):
    """Extract features from images for SMOTE"""
    extractor = FeatureExtractor().to(device)
    extractor.eval()
    
    all_features = []
    all_labels = []
    
    print("Extracting features for SMOTE...")
    with torch.no_grad():
        for images, labels, _ in tqdm(dataloader):
            images = images.to(device)
            features = extractor(images)
            
            all_features.append(features.cpu().numpy())
            all_labels.append(labels.numpy())
    
    all_features = np.vstack(all_features)
    all_labels = np.hstack(all_labels)
    
    return all_features, all_labels

def apply_smote_augmentation(train_loader, device):
    """Apply SMOTE to balance the dataset"""
    print("\n" + "="*50)
    print("APPLYING SMOTE FOR CLASS BALANCING")
    print("="*50)
    
    # Extract features
    features, labels = extract_features_for_smote(train_loader, device)
    
    # Check class distribution before SMOTE
    unique, counts = np.unique(labels, return_counts=True)
    print(f"\nBefore SMOTE:")
    print(f"  Class 0 (Non-Autistic): {counts[0]}")
    print(f"  Class 1 (Autistic): {counts[1]}")
    print(f"  Imbalance ratio: {max(counts)/min(counts):.2f}")
    
    # Apply SMOTE
    smote = SMOTE(random_state=config.SEED, k_neighbors=5)
    features_resampled, labels_resampled = smote.fit_resample(features, labels)
    
    # Check class distribution after SMOTE
    unique, counts = np.unique(labels_resampled, return_counts=True)
    print(f"\nAfter SMOTE:")
    print(f"  Class 0 (Non-Autistic): {counts[0]}")
    print(f"  Class 1 (Autistic): {counts[1]}")
    print(f"  Imbalance ratio: {max(counts)/min(counts):.2f}")
    
    # Visualize
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # Before SMOTE
    unique_orig, counts_orig = np.unique(labels, return_counts=True)
    ax1.bar(['Non-Autistic', 'Autistic'], counts_orig, color=['#4ECDC4', '#FF6B6B'])
    ax1.set_title('Before SMOTE', fontsize=14, fontweight='bold')
    ax1.set_ylabel('Number of Samples')
    for i, v in enumerate(counts_orig):
        ax1.text(i, v + 10, str(v), ha='center', fontweight='bold')
    
    # After SMOTE
    ax2.bar(['Non-Autistic', 'Autistic'], counts, color=['#4ECDC4', '#FF6B6B'])
    ax2.set_title('After SMOTE', fontsize=14, fontweight='bold')
    ax2.set_ylabel('Number of Samples')
    for i, v in enumerate(counts):
        ax2.text(i, v + 10, str(v), ha='center', fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/smote_balancing.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return features_resampled, labels_resampled

# Note: SMOTE is applied on features, not directly on images
# We'll use class weights in loss function as a practical alternative
# Calculate class weights for weighted loss
train_labels = [label for _, label, _ in train_dataset]
class_counts = np.bincount(train_labels)
class_weights = torch.FloatTensor([1.0 / count for count in class_counts]).to(device)
class_weights = class_weights / class_weights.sum() * len(class_weights)

print(f"\nClass weights calculated: {class_weights}")
print("These will be used in the loss function for handling class imbalance")

In [ ]:
#Self-Supervised Pre-training with SimCLR

In [ ]:
class SimCLRAugmentation:
    """SimCLR augmentation strategy"""
    @staticmethod
    def get_simclr_transforms():
        return A.Compose([
            A.Resize(config.IMG_SIZE, config.IMG_SIZE),
            A.HorizontalFlip(p=0.5),
            A.RandomBrightnessContrast(brightness_limit=0.4, contrast_limit=0.4, p=0.8),
            A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=50, val_shift_limit=30, p=0.8),
            A.GaussianBlur(blur_limit=(3, 7), p=0.5),
            A.GaussNoise(var_limit=(10.0, 50.0), p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

class SimCLRDataset(Dataset):
    """Dataset for SimCLR contrastive learning"""
    def __init__(self, root_dir, mode='train'):
        self.root_dir = root_dir
        self.mode = mode
        self.images = []
        self.transform = SimCLRAugmentation.get_simclr_transforms()
        
        self._load_data()
    
    def _load_data(self):
        if self.mode == 'train':
            files = os.listdir(self.root_dir)
            for filename in files:
                if filename.endswith(('.jpg', '.jpeg', '.png')):
                    self.images.append(os.path.join(self.root_dir, filename))
        elif self.mode == 'valid':
            for class_name in ['Autistic', 'Non_Autistic']:
                class_dir = os.path.join(self.root_dir, class_name)
                if os.path.exists(class_dir):
                    for filename in os.listdir(class_dir):
                        if filename.endswith(('.jpg', '.jpeg', '.png')):
                            self.images.append(os.path.join(class_dir, filename))
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        # Create two augmented views
        view1 = self.transform(image=image)['image']
        view2 = self.transform(image=image)['image']
        
        return view1, view2

class ProjectionHead(nn.Module):
    """Projection head for SimCLR"""
    def __init__(self, in_features, hidden_features=512, out_features=224):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_features),
            nn.BatchNorm1d(hidden_features),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_features, out_features)
        )
    
    def forward(self, x):
        return self.net(x)

class SimCLR(nn.Module):
    """SimCLR model for self-supervised pre-training"""
    def __init__(self, base_encoder='resnet50'):
        super().__init__()
        
        # Base encoder
        if base_encoder == 'resnet50':
            self.encoder = models.resnet50(pretrained=False)
            feat_dim = self.encoder.fc.in_features
            self.encoder.fc = nn.Identity()
        
        # Projection head
        self.projection = ProjectionHead(feat_dim, hidden_features=512, out_features=224)
    
    def forward(self, x):
        features = self.encoder(x)
        projections = self.projection(features)
        return features, projections

class NT_Xent_Loss(nn.Module):
    """Normalized Temperature-scaled Cross Entropy Loss for SimCLR"""
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss(reduction="sum")
    
    def forward(self, z_i, z_j):
        batch_size = z_i.shape[0]
        
        # Normalize embeddings
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        
        # Concatenate
        representations = torch.cat([z_i, z_j], dim=0)
        
        # Similarity matrix
        similarity_matrix = F.cosine_similarity(
            representations.unsqueeze(1), 
            representations.unsqueeze(0), 
            dim=2
        )
        
        # Create labels
        labels = torch.arange(batch_size).to(z_i.device)
        labels = torch.cat([labels + batch_size, labels])
        
        # Remove self-similarity
        mask = torch.eye(2 * batch_size, dtype=torch.bool).to(z_i.device)
        similarity_matrix = similarity_matrix.masked_fill(mask, -9e15)
        
        # Scale by temperature
        similarity_matrix = similarity_matrix / self.temperature
        
        # Calculate loss
        loss = self.criterion(similarity_matrix, labels)
        loss = loss / (2 * batch_size)
        
        return loss

def train_simclr(model, train_loader, optimizer, criterion, device, epoch):
    """Train SimCLR for one epoch"""
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc=f'SimCLR Epoch {epoch}')
    for view1, view2 in pbar:
        view1, view2 = view1.to(device), view2.to(device)
        
        # Forward pass
        _, proj1 = model(view1)
        _, proj2 = model(view2)
        
        # Calculate loss
        loss = criterion(proj1, proj2)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        pbar.set_postfix({'loss': loss.item()})
    
    return total_loss / len(train_loader)

def pretrain_with_simclr(train_dir, valid_dir, epochs=20, device=device):
    """Pre-train model using SimCLR"""
    print("\n" + "="*50)
    print("SELF-SUPERVISED PRE-TRAINING WITH SIMCLR")
    print("="*50)
    
    # Create SimCLR datasets
    simclr_train_dataset = SimCLRDataset(train_dir, mode='train')
    simclr_train_loader = DataLoader(
        simclr_train_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=True,
        num_workers=config.NUM_WORKERS,
        pin_memory=True
    )
    
    # Initialize model
    model = SimCLR(base_encoder='resnet50').to(device)
    
    # Optimizer and loss
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=1e-3,
        weight_decay=1e-4
    )
    criterion = NT_Xent_Loss(temperature=0.5)
    
    # Cosine scheduler
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, 
        T_max=epochs
    )
    
    # Training loop
    history = {'loss': []}
    
    for epoch in range(1, epochs + 1):
        loss = train_simclr(model, simclr_train_loader, optimizer, criterion, device, epoch)
        history['loss'].append(loss)
        
        scheduler.step()
        
        print(f"Epoch {epoch}/{epochs} - Loss: {loss:.4f}")
    
    # Save pre-trained encoder
    torch.save(
        model.encoder.state_dict(), 
        f'{config.CHECKPOINT_DIR}/simclr_pretrained_encoder.pth'
    )
    print(f"\nSimCLR pre-trained encoder saved!")
    
    # Plot training curve
    plt.figure(figsize=(10, 5))
    plt.plot(history['loss'], marker='o', linewidth=2, markersize=6)
    plt.title('SimCLR Pre-training Loss', fontsize=14, fontweight='bold')
    plt.xlabel('Epoch')
    plt.ylabel('NT-Xent Loss')
    plt.grid(True, alpha=0.3)
    plt.savefig(f'{config.RESULTS_DIR}/simclr_pretraining.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return model.encoder

# Run SimCLR pre-training (you can adjust epochs based on time constraints)
# For M4 Mac, 10-20 epochs should be reasonable
pretrained_encoder = pretrain_with_simclr(
    config.TRAIN_DIR, 
    config.VALID_DIR, 
    epochs=15,  # Adjust as needed
    device=device
)

print("\nSelf-supervised pre-training completed!")

In [ ]:
#CBAM (Convolutional Block Attention Module)

In [ ]:
class ChannelAttention(nn.Module):
    """Channel attention module"""
    def __init__(self, in_channels, reduction_ratio=16):
        super().__init__()
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        
        self.fc = nn.Sequential(
            nn.Conv2d(in_channels, in_channels // reduction_ratio, 1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels // reduction_ratio, in_channels, 1, bias=False)
        )
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = self.fc(self.avg_pool(x))
        max_out = self.fc(self.max_pool(x))
        out = avg_out + max_out
        return self.sigmoid(out)

class SpatialAttention(nn.Module):
    """Spatial attention module"""
    def __init__(self, kernel_size=7):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size, padding=kernel_size//2, bias=False)
        self.sigmoid = nn.Sigmoid()
    
    def forward(self, x):
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        out = torch.cat([avg_out, max_out], dim=1)
        out = self.conv(out)
        return self.sigmoid(out)

class CBAM(nn.Module):
    """Convolutional Block Attention Module"""
    def __init__(self, in_channels, reduction_ratio=16, kernel_size=7):
        super().__init__()
        self.channel_attention = ChannelAttention(in_channels, reduction_ratio)
        self.spatial_attention = SpatialAttention(kernel_size)
    
    def forward(self, x):
        # Channel attention
        x = x * self.channel_attention(x)
        # Spatial attention
        x = x * self.spatial_attention(x)
        return x

print("CBAM module created successfully!")

In [ ]:
#ResNet152 + CBAM Architecture

In [ ]:
class ResNet152_CBAM(nn.Module):
    """ResNet152 with CBAM attention modules"""
    def __init__(self, num_classes=2, pretrained=True, dropout_rate=0.3, use_simclr_weights=False):
        super().__init__()
        
        # Load base ResNet152
        self.resnet = models.resnet152(pretrained=pretrained)
        
        # Optionally load SimCLR pre-trained weights
        if use_simclr_weights and os.path.exists(f'{config.CHECKPOINT_DIR}/simclr_pretrained_encoder.pth'):
            print("Loading SimCLR pre-trained weights...")
            # Note: SimCLR was trained on ResNet50, so we'll use standard ImageNet weights for ResNet152
            # In practice, you'd want to match architectures
            pass
        
        # Get feature dimensions
        self.feature_dim = self.resnet.fc.in_features
        
        # Remove original FC layer
        self.resnet.fc = nn.Identity()
        
        # Add CBAM modules to layer4 (last residual block)
        self._add_cbam_to_layer4()
        
        # Freeze early layers, fine-tune last N layers
        self._freeze_layers(num_layers_to_finetune=config.RESNET_LAYERS_TO_FINETUNE)
        
        # Global pooling
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        
        # MC Dropout layers
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
    
    def _add_cbam_to_layer4(self):
        """Add CBAM to the last layer"""
        # Get the number of channels in layer4
        layer4_channels = 2048  # ResNet152 layer4 output channels
        
        # Add CBAM after layer4
        self.cbam = CBAM(layer4_channels, reduction_ratio=16)
    
    def _freeze_layers(self, num_layers_to_finetune):
        """Freeze early layers, keep last N layers trainable"""
        # Freeze all first
        for param in self.resnet.parameters():
            param.requires_grad = False
        
        # Unfreeze last layers
        layers_to_train = []
        
        # Always train layer4
        for param in self.resnet.layer4.parameters():
            param.requires_grad = True
        layers_to_train.append('layer4')
        
        # Optionally train layer3
        if num_layers_to_finetune >= 20:
            for param in self.resnet.layer3.parameters():
                param.requires_grad = True
            layers_to_train.append('layer3')
        
        print(f"Fine-tuning layers: {layers_to_train}")
    
    def forward(self, x, return_features=False):
        # ResNet feature extraction
        x = self.resnet.conv1(x)
        x = self.resnet.bn1(x)
        x = self.resnet.relu(x)
        x = self.resnet.maxpool(x)
        
        x = self.resnet.layer1(x)
        x = self.resnet.layer2(x)
        x = self.resnet.layer3(x)
        x = self.resnet.layer4(x)
        
        # Apply CBAM
        x = self.cbam(x)
        
        # Global pooling
        features = self.global_pool(x)
        features = features.view(features.size(0), -1)
        
        # MC Dropout
        features = self.dropout1(features)
        features = self.dropout2(features)
        
        # Classification
        output = self.classifier(features)
        
        if return_features:
            return output, features
        return output
    
    def get_attention_maps(self, x):
        """Get CBAM attention maps for visualization"""
        with torch.no_grad():
            # Forward through ResNet
            x = self.resnet.conv1(x)
            x = self.resnet.bn1(x)
            x = self.resnet.relu(x)
            x = self.resnet.maxpool(x)
            
            x = self.resnet.layer1(x)
            x = self.resnet.layer2(x)
            x = self.resnet.layer3(x)
            x = self.resnet.layer4(x)
            
            # Get channel attention
            ca = self.cbam.channel_attention(x)
            
            # Get spatial attention
            x_ca = x * ca
            sa = self.cbam.spatial_attention(x_ca)
            
            return ca, sa

# Create ResNet152+CBAM model
resnet_cbam = ResNet152_CBAM(
    num_classes=config.NUM_CLASSES,
    pretrained=True,
    dropout_rate=config.DROPOUT_RATE,
    use_simclr_weights=True
).to(device)

# Count parameters
total_params = sum(p.numel() for p in resnet_cbam.parameters())
trainable_params = sum(p.numel() for p in resnet_cbam.parameters() if p.requires_grad)

print(f"\nResNet152+CBAM created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
#Vision Transformer Large (ViT-L)

In [ ]:
class ViTLarge(nn.Module):
    """Vision Transformer Large with fine-tuning"""
    def __init__(self, num_classes=2, pretrained=True, dropout_rate=0.3):
        super().__init__()
        
        # Load pre-trained ViT-Large from timm
        # Using vit_large_patch16_224 (larger model for better performance)
        self.vit = timm.create_model(
            'vit_large_patch16_224',
            pretrained=pretrained,
            num_classes=0  # Remove classification head
        )
        
        # Get feature dimension
        self.feature_dim = self.vit.num_features  # 1024 for ViT-Large
        
        # Freeze early layers, fine-tune last transformer blocks
        self._freeze_layers(num_blocks_to_finetune=4)
        
        # MC Dropout layers
        self.dropout1 = nn.Dropout(dropout_rate)
        self.dropout2 = nn.Dropout(dropout_rate)
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(self.feature_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(512, num_classes)
        )
    
    def _freeze_layers(self, num_blocks_to_finetune=4):
        """Freeze early transformer blocks, keep last N blocks trainable"""
        # Freeze patch embedding and positional encoding
        for param in self.vit.patch_embed.parameters():
            param.requires_grad = False
        
        if hasattr(self.vit, 'pos_embed'):
            self.vit.pos_embed.requires_grad = False
        if hasattr(self.vit, 'cls_token'):
            self.vit.cls_token.requires_grad = False
        
        # Freeze early transformer blocks
        total_blocks = len(self.vit.blocks)
        blocks_to_freeze = total_blocks - num_blocks_to_finetune
        
        for i in range(blocks_to_freeze):
            for param in self.vit.blocks[i].parameters():
                param.requires_grad = False
        
        print(f"ViT-L: Freezing first {blocks_to_freeze} blocks, fine-tuning last {num_blocks_to_finetune} blocks")
    
    def forward(self, x, return_features=False):
        # ViT forward pass
        features = self.vit(x)
        
        # MC Dropout
        features = self.dropout1(features)
        features = self.dropout2(features)
        
        # Classification
        output = self.classifier(features)
        
        if return_features:
            return output, features
        return output

# Create ViT-Large model
vit_large = ViTLarge(
    num_classes=config.NUM_CLASSES,
    pretrained=True,
    dropout_rate=config.DROPOUT_RATE
).to(device)

# Count parameters
total_params = sum(p.numel() for p in vit_large.parameters())
trainable_params = sum(p.numel() for p in vit_large.parameters() if p.requires_grad)

print(f"\nViT-Large created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
#Cross-Attention Fusion Module

In [ ]:
class CrossAttentionFusion(nn.Module):
    """Cross-attention mechanism for fusing CNN and ViT features"""
    def __init__(self, cnn_dim, vit_dim, hidden_dim=512, num_heads=8):
        super().__init__()
        
        self.cnn_dim = cnn_dim
        self.vit_dim = vit_dim
        self.hidden_dim = hidden_dim
        
        # Project features to same dimension
        self.cnn_proj = nn.Linear(cnn_dim, hidden_dim)
        self.vit_proj = nn.Linear(vit_dim, hidden_dim)
        
        # Multi-head cross-attention
        self.cross_attention = nn.MultiheadAttention(
            embed_dim=hidden_dim,
            num_heads=num_heads,
            dropout=0.1,
            batch_first=True
        )
        
        # Layer normalization
        self.norm1 = nn.LayerNorm(hidden_dim)
        self.norm2 = nn.LayerNorm(hidden_dim)
        
        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 4),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim * 4, hidden_dim),
            nn.Dropout(0.1)
        )
        
        # Final fusion
        self.fusion = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(0.1)
        )
    
    def forward(self, cnn_features, vit_features):
        # Project features
        cnn_proj = self.cnn_proj(cnn_features).unsqueeze(1)  # [B, 1, hidden_dim]
        vit_proj = self.vit_proj(vit_features).unsqueeze(1)  # [B, 1, hidden_dim]
        
        # Cross-attention: CNN attends to ViT
        cnn_attended, _ = self.cross_attention(
            query=cnn_proj,
            key=vit_proj,
            value=vit_proj
        )
        cnn_attended = self.norm1(cnn_attended + cnn_proj)
        
        # Cross-attention: ViT attends to CNN
        vit_attended, _ = self.cross_attention(
            query=vit_proj,
            key=cnn_proj,
            value=cnn_proj
        )
        vit_attended = self.norm2(vit_attended + vit_proj)
        
        # Feed-forward
        cnn_attended = cnn_attended + self.ffn(cnn_attended)
        vit_attended = vit_attended + self.ffn(vit_attended)
        
        # Concatenate and fuse
        combined = torch.cat([cnn_attended.squeeze(1), vit_attended.squeeze(1)], dim=1)
        fused = self.fusion(combined)
        
        return fused

print("Cross-Attention Fusion module created successfully!")

In [ ]:
#Hybrid Model (ResNet152+CBAM + ViT-L + Cross-Attention)

In [ ]:
class HybridModel(nn.Module):
    """Hybrid model combining ResNet152+CBAM and ViT-L with cross-attention fusion"""
    def __init__(self, num_classes=2, dropout_rate=0.3):
        super().__init__()
        
        # CNN branch (ResNet152 + CBAM)
        self.cnn_branch = ResNet152_CBAM(
            num_classes=num_classes,
            pretrained=True,
            dropout_rate=dropout_rate,
            use_simclr_weights=True
        )
        # Remove classifier from CNN branch
        cnn_feature_dim = self.cnn_branch.feature_dim
        self.cnn_branch.classifier = nn.Identity()
        
        # ViT branch
        self.vit_branch = ViTLarge(
            num_classes=num_classes,
            pretrained=True,
            dropout_rate=dropout_rate
        )
        # Remove classifier from ViT branch
        vit_feature_dim = self.vit_branch.feature_dim
        self.vit_branch.classifier = nn.Identity()
        
        # Cross-attention fusion
        self.fusion = CrossAttentionFusion(
            cnn_dim=cnn_feature_dim,
            vit_dim=vit_feature_dim,
            hidden_dim=512,
            num_heads=8
        )
        
        # Final classifier
        self.classifier = nn.Sequential(
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x, return_features=False):
        # Extract features from both branches
        cnn_features = self.cnn_branch(x)
        vit_features = self.vit_branch(x)
        
        # Fuse features using cross-attention
        fused_features = self.fusion(cnn_features, vit_features)
        
        # Classification
        output = self.classifier(fused_features)
        
        if return_features:
            return output, fused_features
        return output

# Create hybrid model
hybrid_model = HybridModel(
    num_classes=config.NUM_CLASSES,
    dropout_rate=config.DROPOUT_RATE
).to(device)

# Count parameters
total_params = sum(p.numel() for p in hybrid_model.parameters())
trainable_params = sum(p.numel() for p in hybrid_model.parameters() if p.requires_grad)

print(f"\nHybrid Model created successfully!")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print(f"Frozen parameters: {total_params - trainable_params:,}")

In [ ]:
#Loss Functions

In [ ]:
class FocalLoss(nn.Module):
    """Focal Loss for handling class imbalance"""
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            focal_loss = alpha_t * focal_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

class LabelSmoothingCrossEntropy(nn.Module):
    """Cross-entropy with label smoothing"""
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing
    
    def forward(self, pred, target):
        n_classes = pred.size(-1)
        log_preds = F.log_softmax(pred, dim=-1)
        
        # Create smoothed labels
        with torch.no_grad():
            true_dist = torch.zeros_like(log_preds)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        
        return torch.mean(torch.sum(-true_dist * log_preds, dim=-1))

# Initialize loss functions
if config.USE_FOCAL_LOSS:
    criterion = FocalLoss(alpha=class_weights, gamma=2.0)
    print("Using Focal Loss")
else:
    criterion = LabelSmoothingCrossEntropy(smoothing=config.LABEL_SMOOTHING)
    print(f"Using Label Smoothing Cross-Entropy (smoothing={config.LABEL_SMOOTHING})")

In [ ]:
#Optimizer and Scheduler

In [ ]:
def get_optimizer_and_scheduler(model, train_loader):
    """Create optimizer and learning rate scheduler"""
    
    # Separate parameters by learning rate
    cnn_params = []
    vit_params = []
    fusion_params = []
    classifier_params = []
    
    for name, param in model.named_parameters():
        if not param.requires_grad:
            continue
        
        if 'cnn_branch' in name:
            cnn_params.append(param)
        elif 'vit_branch' in name:
            vit_params.append(param)
        elif 'fusion' in name:
            fusion_params.append(param)
        elif 'classifier' in name:
            classifier_params.append(param)
    
    # Different learning rates for different parts
    param_groups = [
        {'params': cnn_params, 'lr': config.LEARNING_RATE * 0.1, 'name': 'cnn'},
        {'params': vit_params, 'lr': config.LEARNING_RATE * 0.1, 'name': 'vit'},
        {'params': fusion_params, 'lr': config.LEARNING_RATE, 'name': 'fusion'},
        {'params': classifier_params, 'lr': config.LEARNING_RATE, 'name': 'classifier'}
    ]
    
    # AdamW optimizer
    optimizer = torch.optim.AdamW(
        param_groups,
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY,
        betas=(0.9, 0.999)
    )
    
    # Cosine annealing with warm restarts
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer,
        T_0=10,  # Restart every 10 epochs
        T_mult=2,  # Double the period after each restart
        eta_min=1e-6
    )
    
    print("\nOptimizer configuration:")
    for group in param_groups:
        print(f"  {group['name']}: lr={group['lr']}, params={len(group['params'])}")
    
    return optimizer, scheduler

# Create optimizer and scheduler for hybrid model
optimizer, scheduler = get_optimizer_and_scheduler(hybrid_model, train_loader)

In [ ]:
#Training and Validation Functions

In [ ]:
class MetricsTracker:
    """Track training metrics"""
    def __init__(self):
        self.reset()
    
    def reset(self):
        self.losses = []
        self.accuracies = []
        self.precisions = []
        self.recalls = []
        self.f1_scores = []
        self.all_preds = []
        self.all_labels = []
        self.all_probs = []
    
    def update(self, loss, preds, labels, probs=None):
        self.losses.append(loss)
        self.all_preds.extend(preds.cpu().numpy())
        self.all_labels.extend(labels.cpu().numpy())
        if probs is not None:
            self.all_probs.extend(probs.cpu().numpy())
    
    def compute_metrics(self):
        preds = np.array(self.all_preds)
        labels = np.array(self.all_labels)
        
        acc = accuracy_score(labels, preds)
        prec = precision_score(labels, preds, average='binary', zero_division=0)
        rec = recall_score(labels, preds, average='binary', zero_division=0)
        f1 = f1_score(labels, preds, average='binary', zero_division=0)
        
        metrics = {
            'loss': np.mean(self.losses),
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1
        }
        
        if len(self.all_probs) > 0:
            probs = np.array(self.all_probs)
            try:
                auc = roc_auc_score(labels, probs[:, 1])
                metrics['auc'] = auc
            except:
                metrics['auc'] = 0.0
        
        return metrics

def train_epoch(model, train_loader, criterion, optimizer, device, epoch):
    """Train for one epoch"""
    model.train()
    tracker = MetricsTracker()
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch} [Train]')
    for images, labels, _ in pbar:
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        
        # Gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        
        optimizer.step()
        
        # Metrics
        probs = F.softmax(outputs, dim=1)
        preds = torch.argmax(probs, dim=1)
        tracker.update(loss.item(), preds, labels, probs)
        
        # Update progress bar
        pbar.set_postfix({'loss': loss.item()})
    
    return tracker.compute_metrics()

def validate_epoch(model, valid_loader, criterion, device, epoch):
    """Validate for one epoch"""
    model.eval()
    tracker = MetricsTracker()
    
    with torch.no_grad():
        pbar = tqdm(valid_loader, desc=f'Epoch {epoch} [Valid]')
        for images, labels, _ in pbar:
            images, labels = images.to(device), labels.to(device)
            
            # Forward pass
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            # Metrics
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            tracker.update(loss.item(), preds, labels, probs)
            
            pbar.set_postfix({'loss': loss.item()})
    
    return tracker.compute_metrics()

print("Training functions created successfully!")

In [ ]:
#Complete Training Loop with Early Stopping

In [ ]:
### import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# =======================
# Metrics Tracker
# =======================
class MetricsTracker:
    def __init__(self):
        self.reset()

    def reset(self):
        self.losses = []
        self.all_preds = []
        self.all_labels = []
        self.all_probs = []

    def update(self, loss, preds, labels, probs=None):
        self.losses.append(loss)
        self.all_preds.extend(preds.detach().cpu().numpy())
        self.all_labels.extend(labels.detach().cpu().numpy())
        if probs is not None:
            self.all_probs.extend(probs.detach().cpu().numpy())  # <-- FIXED

    def compute(self):
        acc = accuracy_score(self.all_labels, self.all_preds)
        prec = precision_score(self.all_labels, self.all_preds, average="weighted", zero_division=0)
        rec = recall_score(self.all_labels, self.all_preds, average="weighted", zero_division=0)
        f1 = f1_score(self.all_labels, self.all_preds, average="weighted", zero_division=0)
        return {
            "loss": sum(self.losses) / len(self.losses),
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1": f1,
        }

# =======================
# Training Epoch
# =======================
def train_epoch(model, train_loader, criterion, optimizer, device, epoch):
    model.train()
    tracker = MetricsTracker()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch} [Train]")

    for images, labels, _ in pbar:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        with torch.no_grad():
            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            tracker.update(loss.item(), preds, labels, probs)

        pbar.set_postfix({"loss": loss.item()})

    return tracker.compute()

# =======================
# Validation Epoch
# =======================
def validate_epoch(model, valid_loader, criterion, device, epoch):
    model.eval()
    tracker = MetricsTracker()
    pbar = tqdm(valid_loader, desc=f"Epoch {epoch} [Valid]")

    with torch.no_grad():
        for images, labels, _ in pbar:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            probs = F.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            tracker.update(loss.item(), preds, labels, probs)

            pbar.set_postfix({"loss": loss.item()})

    return tracker.compute()

# =======================
# Early Stopping
# =======================
class EarlyStopping:
    """Early stopping to prevent overfitting"""
    def __init__(self, patience=10, min_delta=0.001, mode="max"):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0

    def __call__(self, score, epoch):
        if self.best_score is None:
            self.best_score = score
            self.best_epoch = epoch
            return False

        if self.mode == "max":
            if score > self.best_score + self.min_delta:
                self.best_score = score
                self.best_epoch = epoch
                self.counter = 0
            else:
                self.counter += 1
        else:  # "min"
            if score < self.best_score - self.min_delta:
                self.best_score = score
                self.best_epoch = epoch
                self.counter = 0
            else:
                self.counter += 1

        if self.counter >= self.patience:
            self.early_stop = True
            return True

        return False

# =======================
# Training Loop
# =======================
def train_model(model, train_loader, valid_loader, criterion, optimizer, scheduler,
                num_epochs, device, model_name="hybrid", checkpoint_dir="./checkpoints"):
    """Complete training loop"""
    print("\n" + "=" * 70)
    print(f"TRAINING {model_name.upper()} MODEL")
    print("=" * 70)

    history = {
        "train_loss": [], "train_acc": [], "train_f1": [],
        "valid_loss": [], "valid_acc": [], "valid_f1": [],
        "learning_rates": []
    }

    best_valid_f1 = 0.0
    best_model_state = None
    early_stopping = EarlyStopping(patience=15, min_delta=0.001, mode="max")

    for epoch in range(1, num_epochs + 1):
        # Train
        train_metrics = train_epoch(model, train_loader, criterion, optimizer, device, epoch)

        # Validate
        valid_metrics = validate_epoch(model, valid_loader, criterion, device, epoch)

        # Scheduler step
        scheduler.step()

        # Store history
        history["train_loss"].append(train_metrics["loss"])
        history["train_acc"].append(train_metrics["accuracy"])
        history["train_f1"].append(train_metrics["f1"])
        history["valid_loss"].append(valid_metrics["loss"])
        history["valid_acc"].append(valid_metrics["accuracy"])
        history["valid_f1"].append(valid_metrics["f1"])
        history["learning_rates"].append(optimizer.param_groups[0]["lr"])

        # Print metrics
        print(f"\nEpoch {epoch}/{num_epochs}")
        print(f"  Train - Loss: {train_metrics['loss']:.4f}, Acc: {train_metrics['accuracy']:.4f}, "
              f"Prec: {train_metrics['precision']:.4f}, Rec: {train_metrics['recall']:.4f}, F1: {train_metrics['f1']:.4f}")
        print(f"  Valid - Loss: {valid_metrics['loss']:.4f}, Acc: {valid_metrics['accuracy']:.4f}, "
              f"Prec: {valid_metrics['precision']:.4f}, Rec: {valid_metrics['recall']:.4f}, F1: {valid_metrics['f1']:.4f}")
        print(f"  LR: {optimizer.param_groups[0]['lr']:.6f}")

        # Save best model
        if valid_metrics["f1"] > best_valid_f1:
            best_valid_f1 = valid_metrics["f1"]
            best_model_state = model.state_dict().copy()
            torch.save({
                "epoch": epoch,
                "model_state_dict": best_model_state,
                "optimizer_state_dict": optimizer.state_dict(),
                "valid_f1": best_valid_f1,
                "history": history
            }, f"{checkpoint_dir}/{model_name}_best.pth")
            print(f"  ✓ Best model saved! (F1: {best_valid_f1:.4f})")

        # Early stopping
        if early_stopping(valid_metrics["f1"], epoch):
            print(f"\n⚠ Early stopping triggered at epoch {epoch}")
            print(f"  Best epoch was {early_stopping.best_epoch} with F1: {early_stopping.best_score:.4f}")
            break

    # Load best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, history

#Train the hybrid model
trained_hybrid_model, hybrid_history = train_model(
    model=hybrid_model,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=optimizer,
    scheduler=scheduler,
    num_epochs=config.EPOCHS,
    device=device,
    model_name='hybrid',
    checkpoint_dir="./checkpoints"
)
# trained_model, history = train_model(
#     model=hybrid_model,
#     train_loader=train_loader,
#     valid_loader=valid_loader,
#     criterion=criterion,
#     optimizer=optimizer,
#     scheduler=scheduler,
#     num_epochs=50,
#     device=device,
#     model_name="hybrid",
#     checkpoint_dir="./checkpoints"
# )

print("\n✓ Hybrid model training completed!")

In [ ]:
#Visualize Training History

In [ ]:
def plot_training_history(history, model_name='Hybrid'):
    """Plot comprehensive training history"""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    
    epochs = range(1, len(history['train_loss']) + 1)
    
    # Loss
    axes[0, 0].plot(epochs, history['train_loss'], 'b-o', label='Train Loss', linewidth=2, markersize=4)
    axes[0, 0].plot(epochs, history['valid_loss'], 'r-s', label='Valid Loss', linewidth=2, markersize=4)
    axes[0, 0].set_title('Loss Over Epochs', fontsize=14, fontweight='bold')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[0, 1].plot(epochs, history['train_acc'], 'b-o', label='Train Accuracy', linewidth=2, markersize=4)
    axes[0, 1].plot(epochs, history['valid_acc'], 'r-s', label='Valid Accuracy', linewidth=2, markersize=4)
    axes[0, 1].set_title('Accuracy Over Epochs', fontsize=14, fontweight='bold')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # F1 Score
    axes[1, 0].plot(epochs, history['train_f1'], 'b-o', label='Train F1', linewidth=2, markersize=4)
    axes[1, 0].plot(epochs, history['valid_f1'], 'r-s', label='Valid F1', linewidth=2, markersize=4)
    axes[1, 0].set_title('F1 Score Over Epochs', fontsize=14, fontweight='bold')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('F1 Score')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[1, 1].plot(epochs, history['learning_rates'], 'g-o', linewidth=2, markersize=4)
    axes[1, 1].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[1, 1].set_xlabel('Epoch')
    axes[1, 1].set_ylabel('Learning Rate')
    axes[1, 1].set_yscale('log')
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/{model_name.lower()}_training_history.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot training history
plot_training_history(hybrid_history, model_name='Hybrid')

In [ ]:
#Create Additional Models for Ensemble

In [ ]:
# Train ResNet152+CBAM alone
print("\n" + "="*70)
print("TRAINING RESNET152+CBAM MODEL (for ensemble)")
print("="*70)

resnet_optimizer, resnet_scheduler = get_optimizer_and_scheduler(resnet_cbam, train_loader)

trained_resnet_cbam, resnet_history = train_model(
    model=resnet_cbam,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=resnet_optimizer,
    scheduler=resnet_scheduler,
    num_epochs=config.EPOCHS,
    device=device,
    model_name='resnet_cbam'
)

# Plot ResNet training history
plot_training_history(resnet_history, model_name='ResNet152_CBAM')

# Train ViT-Large alone
print("\n" + "="*70)
print("TRAINING VIT-LARGE MODEL (for ensemble)")
print("="*70)

vit_optimizer, vit_scheduler = get_optimizer_and_scheduler(vit_large, train_loader)

trained_vit_large, vit_history = train_model(
    model=vit_large,
    train_loader=train_loader,
    valid_loader=valid_loader,
    criterion=criterion,
    optimizer=vit_optimizer,
    scheduler=vit_scheduler,
    num_epochs=config.EPOCHS,
    device=device,
    model_name='vit_large'
)

# Plot ViT training history
plot_training_history(vit_history, model_name='ViT_Large')

print("\n✓ All models trained for ensemble!")

In [ ]:
#Test-Time Augmentation (TTA)

In [ ]:
def predict_with_tta(model, image, transforms_list, device):
    """Predict with test-time augmentation"""
    model.eval()
    predictions = []
    
    with torch.no_grad():
        for transform in transforms_list:
            augmented = transform(image=image)['image'].unsqueeze(0).to(device)

            # Ensure correct input size for ViT
            if augmented.shape[-1] != 224 or augmented.shape[-2] != 224:
                augmented = F.interpolate(augmented, size=(224, 224), mode='bilinear', align_corners=False)

            output = model(augmented)
            prob = F.softmax(output, dim=1)
            predictions.append(prob.cpu().numpy())
    
    avg_prediction = np.mean(predictions, axis=0)
    return avg_prediction


def evaluate_with_tta(model, test_loader, device, model_name='model'):
    """Evaluate model with TTA"""
    print(f"\n{'='*70}")
    print(f"EVALUATING {model_name.upper()} WITH TEST-TIME AUGMENTATION")
    print('='*70)
    
    tta_transforms = AutoAugmentTransforms.get_tta_transforms()
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    for images, labels, paths in tqdm(test_loader, desc='TTA Prediction'):
        for i in range(images.size(0)):
            # Get single image
            img_path = paths[i]
            image = cv2.imread(img_path)
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
            
            # Predict with TTA
            prob = predict_with_tta(model, image, tta_transforms, device)
            pred = np.argmax(prob[0])
            all_preds.append(pred)
            all_labels.append(labels[i].item())
            all_probs.append(prob[0])
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary')
    recall = recall_score(all_labels, all_preds, average='binary')
    f1 = f1_score(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs[:, 1])
    
    print(f"\n{model_name} Results with TTA:")
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  AUC:       {auc:.4f}")
    
    return {
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

# Evaluate each model with TTA
hybrid_tta_results = evaluate_with_tta(trained_hybrid_model, test_loader, device, 'Hybrid Model')
resnet_tta_results = evaluate_with_tta(trained_resnet_cbam, test_loader, device, 'ResNet152+CBAM')
vit_tta_results = evaluate_with_tta(trained_vit_large, test_loader, device, 'ViT-Large')

In [ ]:
#Ensemble Model with Weighted Voting

In [ ]:
class EnsembleModel:
    """Ensemble of multiple models with weighted voting"""
    def __init__(self, models, weights=None, device='cpu'):
        self.models = models
        self.device = device
        
        # Default: equal weights
        if weights is None:
            self.weights = [1.0 / len(models)] * len(models)
        else:
            # Normalize weights
            total = sum(weights)
            self.weights = [w / total for w in weights]
        
        # Set all models to eval mode
        for model in self.models:
            model.eval()
    
    def predict(self, images):
        """Ensemble prediction"""
        all_probs = []
        
        with torch.no_grad():
            for model, weight in zip(self.models, self.weights):
                outputs = model(images)
                probs = F.softmax(outputs, dim=1)
                all_probs.append(probs * weight)
        
        # Weighted average
        ensemble_probs = torch.stack(all_probs).sum(dim=0)
        return ensemble_probs
    
    def predict_with_tta(self, image, transforms_list):
        """Ensemble prediction with TTA"""
        tta_predictions = []
        
        for transform in transforms_list:
            augmented = transform(image=image)['image']
            
            # Enforce 224x224 for all models (esp. ViT)
            if augmented.shape[-2:] != (224, 224):
                augmented = F.interpolate(augmented.unsqueeze(0), size=(224, 224), mode='bilinear', align_corners=False).squeeze(0)
            
            augmented = augmented.unsqueeze(0).to(self.device)
            probs = self.predict(augmented)
            tta_predictions.append(probs.cpu().numpy())
        
        avg_prediction = np.mean(tta_predictions, axis=0)
        return avg_prediction


def evaluate_ensemble(ensemble, test_loader, device, use_tta=True):
    """Evaluate ensemble model"""
    print("\n" + "="*70)
    print("EVALUATING ENSEMBLE MODEL" + (" WITH TTA" if use_tta else ""))
    print("="*70)
    
    all_preds = []
    all_labels = []
    all_probs = []
    
    if use_tta:
        tta_transforms = AutoAugmentTransforms.get_tta_transforms()
        
        for images, labels, paths in tqdm(test_loader, desc='Ensemble + TTA'):
            for i in range(images.size(0)):
                img_path = paths[i]
                image = cv2.imread(img_path)
                image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
                
                prob = ensemble.predict_with_tta(image, tta_transforms)
                pred = np.argmax(prob[0])
                
                all_preds.append(pred)
                all_labels.append(labels[i].item())
                all_probs.append(prob[0])
    else:
        with torch.no_grad():
            for images, labels, _ in tqdm(test_loader, desc='Ensemble'):
                images = images.to(device)
                probs = ensemble.predict(images)
                preds = torch.argmax(probs, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.numpy())
                all_probs.extend(probs.cpu().numpy())
    
    # Calculate metrics
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    all_probs = np.array(all_probs)
    
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary')
    recall = recall_score(all_labels, all_preds, average='binary')
    f1 = f1_score(all_labels, all_preds, average='binary')
    auc = roc_auc_score(all_labels, all_probs[:, 1])
    
    print(f"\nEnsemble Results:")
    print(f"  Accuracy:  {accuracy:.4f} ({accuracy*100:.2f}%)")
    print(f"  Precision: {precision:.4f}")
    print(f"  Recall:    {recall:.4f}")
    print(f"  F1 Score:  {f1:.4f}")
    print(f"  AUC:       {auc:.4f}")
    
    return {
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs,
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'auc': auc
    }

# Create ensemble with learned weights (based on validation F1 scores)
# You can adjust these weights based on individual model performance
ensemble_models = [trained_hybrid_model, trained_resnet_cbam, trained_vit_large]
ensemble_weights = [0.5, 0.25, 0.25]  # Hybrid gets more weight

ensemble = EnsembleModel(
    models=ensemble_models,
    weights=ensemble_weights,
    device=device
)

# Evaluate ensemble
ensemble_results = evaluate_ensemble(ensemble, test_loader, device, use_tta=True)

In [ ]:
#Confusion Matrix and Classification Report

In [ ]:
def plot_confusion_matrix(y_true, y_pred, class_names, title='Confusion Matrix', save_name='confusion_matrix'):
    """Plot confusion matrix with detailed annotations"""
    cm = confusion_matrix(y_true, y_pred)
    
    # Calculate percentages
    cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100
    
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Create heatmap
    sns.heatmap(cm, annot=False, fmt='d', cmap='Blues', cbar=True, 
                square=True, linewidths=2, linecolor='white',
                xticklabels=class_names, yticklabels=class_names, ax=ax)
    
    # Add annotations with counts and percentages
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            text = f'{cm[i, j]}\n({cm_percent[i, j]:.1f}%)'
            color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
            ax.text(j + 0.5, i + 0.5, text, ha='center', va='center', 
                   color=color, fontsize=14, fontweight='bold')
    
    ax.set_title(title, fontsize=16, fontweight='bold', pad=20)
    ax.set_ylabel('True Label', fontsize=14, fontweight='bold')
    ax.set_xlabel('Predicted Label', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/{save_name}.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    # Print classification report
    print(f"\nClassification Report for {title}:")
    print("="*70)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

# Plot confusion matrices for all models
class_names = ['Non-Autistic', 'Autistic']

print("\n" + "="*70)
print("CONFUSION MATRICES AND CLASSIFICATION REPORTS")
print("="*70)

# Hybrid Model
plot_confusion_matrix(
    hybrid_tta_results['labels'], 
    hybrid_tta_results['predictions'],
    class_names,
    title='Hybrid Model (with TTA)',
    save_name='cm_hybrid_tta'
)

# ResNet152+CBAM
plot_confusion_matrix(
    resnet_tta_results['labels'], 
    resnet_tta_results['predictions'],
    class_names,
    title='ResNet152+CBAM (with TTA)',
    save_name='cm_resnet_tta'
)

# ViT-Large
plot_confusion_matrix(
    vit_tta_results['labels'], 
    vit_tta_results['predictions'],
    class_names,
    title='ViT-Large (with TTA)',
    save_name='cm_vit_tta'
)

# Ensemble
plot_confusion_matrix(
    ensemble_results['labels'], 
    ensemble_results['predictions'],
    class_names,
    title='Ensemble Model (with TTA)',
    save_name='cm_ensemble_tta'
)

In [ ]:
#ROC Curves and AUC Comparison

In [ ]:
from sklearn.metrics import roc_curve, auc

def plot_roc_curves(results_dict, save_name='roc_curves'):
    """Plot ROC curves for multiple models"""
    plt.figure(figsize=(12, 8))
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4', '#FFEAA7']
    
    for idx, (model_name, results) in enumerate(results_dict.items()):
        fpr, tpr, _ = roc_curve(results['labels'], results['probabilities'][:, 1])
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, color=colors[idx], linewidth=3, 
                label=f'{model_name} (AUC = {roc_auc:.4f})')
    
    # Plot diagonal
    plt.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Random Classifier')
    
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=14, fontweight='bold')
    plt.ylabel('True Positive Rate', fontsize=14, fontweight='bold')
    plt.title('ROC Curves - Model Comparison', fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc="lower right", fontsize=12, frameon=True, shadow=True)
    plt.grid(True, alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/{save_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

# Create ROC curve comparison
roc_results = {
    'Hybrid Model': hybrid_tta_results,
    'ResNet152+CBAM': resnet_tta_results,
    'ViT-Large': vit_tta_results,
    'Ensemble': ensemble_results
}

plot_roc_curves(roc_results)

In [ ]:
#Comprehensive Results Comparison

In [ ]:
def create_results_comparison_table(results_dict):
    """Create comprehensive comparison table"""
    
    # Create DataFrame
    data = []
    for model_name, results in results_dict.items():
        data.append({
            'Model': model_name,
            'Accuracy': f"{results['accuracy']:.4f}",
            'Precision': f"{results['precision']:.4f}",
            'Recall': f"{results['recall']:.4f}",
            'F1 Score': f"{results['f1']:.4f}",
            'AUC': f"{results['auc']:.4f}",
            'Accuracy %': f"{results['accuracy']*100:.2f}%"
        })
    
    df = pd.DataFrame(data)
    
    # Display table
    print("\n" + "="*90)
    print("COMPREHENSIVE RESULTS COMPARISON")
    print("="*90)
    print(df.to_string(index=False))
    print("="*90)
    
    # Save to CSV
    df.to_csv(f'{config.RESULTS_DIR}/results_comparison.csv', index=False)
    print(f"\nResults saved to: {config.RESULTS_DIR}/results_comparison.csv")
    
    return df

# Create comparison
results_comparison = {
    'Hybrid Model (TTA)': hybrid_tta_results,
    'ResNet152+CBAM (TTA)': resnet_tta_results,
    'ViT-Large (TTA)': vit_tta_results,
    'Ensemble (TTA)': ensemble_results
}

comparison_df = create_results_comparison_table(results_comparison)

In [ ]:
#Visualize Metrics Comparison

In [ ]:
def plot_metrics_comparison(comparison_df, save_name='metrics_comparison'):
    """Plot bar charts comparing all metrics"""
    
    # Prepare data
    models = comparison_df['Model'].tolist()
    metrics = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'AUC']
    
    # Convert string percentages to float
    data = {}
    for metric in metrics:
        data[metric] = [float(comparison_df[comparison_df['Model'] == model][metric].values[0]) 
                       for model in models]
    
    # Create subplots
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    axes = axes.flatten()
    
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#96CEB4']
    
    for idx, metric in enumerate(metrics):
        ax = axes[idx]
        bars = ax.bar(range(len(models)), data[metric], color=colors, alpha=0.8, edgecolor='black', linewidth=1.5)
        
        ax.set_title(metric, fontsize=14, fontweight='bold')
        ax.set_ylabel('Score', fontsize=12)
        ax.set_xticks(range(len(models)))
        ax.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
        ax.set_ylim([0.7, 1.0])
        ax.grid(True, alpha=0.3, axis='y')
        
        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.4f}',
                   ha='center', va='bottom', fontweight='bold', fontsize=10)
    
    # Overall comparison in last subplot
    ax = axes[5]
    x = np.arange(len(models))
    width = 0.15
    
    for idx, metric in enumerate(metrics):
        offset = (idx - 2) * width
        ax.bar(x + offset, data[metric], width, label=metric, alpha=0.8)
    
    ax.set_title('Overall Comparison', fontsize=14, fontweight='bold')
    ax.set_ylabel('Score', fontsize=12)
    ax.set_xticks(x)
    ax.set_xticklabels(models, rotation=45, ha='right', fontsize=10)
    ax.set_ylim([0.7, 1.0])
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/{save_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

# Plot comparison
plot_metrics_comparison(comparison_df)

In [ ]:
#Attention Visualization (Grad-CAM)

In [ ]:
import torch.nn.functional as F
from torch.autograd import Variable

class GradCAM:
    """Gradient-weighted Class Activation Mapping"""
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradients = None
        self.activations = None
        
        # Register hooks
        target_layer.register_forward_hook(self.save_activation)
        target_layer.register_backward_hook(self.save_gradient)
    
    def save_activation(self, module, input, output):
        self.activations = output.detach()
    
    def save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()
    
    def generate_cam(self, input_image, target_class=None):
        """Generate CAM for input image"""
        self.model.eval()
        
        # Forward pass
        output = self.model(input_image)
        
        if target_class is None:
            target_class = output.argmax(dim=1).item()
        
        # Backward pass
        self.model.zero_grad()
        output[0, target_class].backward()
        
        # Calculate weights
        weights = torch.mean(self.gradients, dim=(2, 3), keepdim=True)
        
        # Generate CAM
        cam = torch.sum(weights * self.activations, dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=input_image.shape[2:], mode='bilinear', align_corners=False)
        
        # Normalize
        cam = cam - cam.min()
        cam = cam / cam.max()
        
        return cam.squeeze().cpu().numpy()

def visualize_grad_cam(model, test_loader, num_samples=6, save_name='grad_cam_examples'):
    """Visualize Grad-CAM for sample images"""
    
    # Get target layer (last conv layer for ResNet)
    if hasattr(model, 'cnn_branch'):
        target_layer = model.cnn_branch.resnet.layer4[-1].conv3
    elif hasattr(model, 'resnet'):
        target_layer = model.resnet.layer4[-1].conv3
    else:
        print("Model architecture not supported for Grad-CAM")
        return
    
    grad_cam = GradCAM(model, target_layer)
    
    # Get sample images
    model.eval()
    samples = []
    
    for images, labels, paths in test_loader:
        for i in range(min(num_samples - len(samples), images.size(0))):
            samples.append((images[i], labels[i], paths[i]))
        
        if len(samples) >= num_samples:
            break
    
    # Visualize
    fig, axes = plt.subplots(num_samples, 3, figsize=(12, 4*num_samples))
    
    for idx, (image, label, path) in enumerate(samples):
        # Original image
        img = cv2.imread(path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img_resized = cv2.resize(img, (224, 224))
        
        # Generate CAM
        image_tensor = image.unsqueeze(0).to(device)
        cam = grad_cam.generate_cam(image_tensor)
        
        # Get prediction
        with torch.no_grad():
            output = model(image_tensor)
            prob = F.softmax(output, dim=1)
            pred = output.argmax(dim=1).item()
            confidence = prob[0, pred].item()
        
        # Plot original
        axes[idx, 0].imshow(img_resized)
        axes[idx, 0].set_title(f'Original\nTrue: {class_names[label]}', fontweight='bold')
        axes[idx, 0].axis('off')
        
        # Plot CAM
        axes[idx, 1].imshow(cam, cmap='jet')
        axes[idx, 1].set_title('Grad-CAM Heatmap', fontweight='bold')
        axes[idx, 1].axis('off')
        
        # Plot overlay
        heatmap = cv2.applyColorMap(np.uint8(255 * cam), cv2.COLORMAP_JET)
        heatmap = cv2.cvtColor(heatmap, cv2.COLOR_BGR2RGB)
        overlay = cv2.addWeighted(img_resized, 0.6, heatmap, 0.4, 0)
        
        axes[idx, 2].imshow(overlay)
        axes[idx, 2].set_title(f'Overlay\nPred: {class_names[pred]} ({confidence:.2%})', fontweight='bold')
        axes[idx, 2].axis('off')
    
    plt.tight_layout()
    plt.savefig(f'{config.RESULTS_DIR}/{save_name}.png', dpi=300, bbox_inches='tight')
    plt.show()

# Visualize Grad-CAM
visualize_grad_cam(trained_hybrid_model, test_loader, num_samples=6)

In [ ]:
#Final Summary Report

In [ ]:
def generate_final_report(comparison_df, hybrid_history, best_model_name='Ensemble'):
    """Generate comprehensive final report"""
    
    print("\n" + "="*90)
    print(" "*30 + "FINAL SUMMARY REPORT")
    print("="*90)
    
    # Best model
    best_model = comparison_df.loc[comparison_df['Model'] == f'{best_model_name} (TTA)']
    
    print("\n🏆 BEST PERFORMING MODEL")
    print("-"*90)
    print(f"Model: {best_model_name} with Test-Time Augmentation")
    print(f"  • Accuracy:  {best_model['Accuracy %'].values[0]}")
    print(f"  • Precision: {best_model['Precision'].values[0]}")
    print(f"  • Recall:    {best_model['Recall'].values[0]}")
    print(f"  • F1 Score:  {best_model['F1 Score'].values[0]}")
    print(f"  • AUC:       {best_model['AUC'].values[0]}")
    
    # Training summary
    print("\n📊 TRAINING SUMMARY")
    print("-"*90)
    print(f"Total Epochs Trained: {len(hybrid_history['train_loss'])}")
    print(f"Best Training Accuracy: {max(hybrid_history['train_acc']):.4f}")
    print(f"Best Validation Accuracy: {max(hybrid_history['valid_acc']):.4f}")
    print(f"Final Training Loss: {hybrid_history['train_loss'][-1]:.4f}")
    print(f"Final Validation Loss: {hybrid_history['valid_loss'][-1]:.4f}")
    
    # Improvements over base paper
    base_paper_acc = 0.9133  # 91.33% from the paper
    improvement = (float(best_model['Accuracy'].values[0]) - base_paper_acc) * 100
    
    print("\n📈 IMPROVEMENT OVER BASE PAPER")
    print("-"*90)
    print(f"Base Paper Accuracy: 91.33%")
    print(f"Our Best Accuracy: {best_model['Accuracy %'].values[0]}")
    print(f"Improvement: {improvement:+.2f} percentage points")
    
    
    # Key techniques used
    print("\n🔧 KEY TECHNIQUES IMPLEMENTED")
    print("-"*90)
    print("  ✓ Advanced Data Augmentation (AutoAugment)")
    print("  ✓ Self-Supervised Pre-training (SimCLR)")
    print("  ✓ ResNet152 + CBAM Attention")
    print("  ✓ Vision Transformer Large (ViT-L)")
    print("  ✓ Cross-Attention Fusion")
    print("  ✓ AdamW Optimizer + Cosine Annealing")
    print("  ✓ Focal Loss for Class Imbalance")
    print("  ✓ Test-Time Augmentation (TTA)")
    print("  ✓ Ensemble of 3 Models")
    print("  ✓ Monte Carlo Dropout for Uncertainty")
    
    # Dataset info
    print("\n📁 DATASET INFORMATION")
    print("-"*90)
    print(f"Training Samples: {len(train_dataset)}")
    print(f"Validation Samples: {len(valid_dataset)}")
    print(f"Test Samples: {len(test_dataset)}")
    print(f"Image Size: {config.IMG_SIZE}x{config.IMG_SIZE}")
    print(f"Batch Size: {config.BATCH_SIZE}")
    
    # Saved files
    print("\n💾 SAVED FILES")
    print("-"*90)
    print(f"  • Model Checkpoints: {config.CHECKPOINT_DIR}/")
    print(f"  • Results & Plots: {config.RESULTS_DIR}/")
    print(f"  • Comparison Table: {config.RESULTS_DIR}/results_comparison.csv")
    
    print("\n" + "="*90)
    print("Report generation complete!")
    print("="*90 + "\n")
    
    # Save report to file
    report_path = f'{config.RESULTS_DIR}/final_report.txt'
    with open(report_path, 'w') as f:
        f.write("="*90 + "\n")
        f.write(" "*30 + "FINAL SUMMARY REPORT\n")
        f.write("="*90 + "\n\n")
        f.write(f"Best Model: {best_model_name}\n")
        f.write(f"Accuracy: {best_model['Accuracy %'].values[0]}\n")
        f.write(f"F1 Score: {best_model['F1 Score'].values[0]}\n")
        f.write(f"Improvement over base: {improvement:+.2f}%\n")
    
    print(f"Report saved to: {report_path}")

# Generate final report
generate_final_report(comparison_df, hybrid_history, best_model_name='Ensemble')

In [ ]:
#Save All Models and Create Inference Function

In [ ]:
def save_all_models():
    """Save all trained models"""
    print("\n" + "="*70)
    print("SAVING ALL MODELS")
    print("="*70)
    
    # Save individual models
    torch.save({
        'model_state_dict': trained_hybrid_model.state_dict(),
        'model_type': 'hybrid',
        'config': vars(config)
    }, f'{config.CHECKPOINT_DIR}/hybrid_final.pth')
    print("✓ Hybrid model saved")
    
    torch.save({
        'model_state_dict': trained_resnet_cbam.state_dict(),
        'model_type': 'resnet_cbam',
        'config': vars(config)
    }, f'{config.CHECKPOINT_DIR}/resnet_cbam_final.pth')
    print("✓ ResNet152+CBAM saved")
    
    torch.save({
        'model_state_dict': trained_vit_large.state_dict(),
        'model_type': 'vit_large',
        'config': vars(config)
    }, f'{config.CHECKPOINT_DIR}/vit_large_final.pth')
    print("✓ ViT-Large saved")
    
    # Save ensemble configuration
    ensemble_config = {
        'weights': ensemble.weights,
        'model_types': ['hybrid', 'resnet_cbam', 'vit_large']
    }
    torch.save(ensemble_config, f'{config.CHECKPOINT_DIR}/ensemble_config.pth')
    print("✓ Ensemble configuration saved")
    
    print("\nAll models saved successfully!")

# Save models
save_all_models()

# Inference function for new images
def predict_single_image(image_path, model, use_tta=True, device=device):
    """Predict autism from a single image"""
    model.eval()

    # Load and preprocess image
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    def resize_if_needed(tensor, size=(224, 224)):
        """Force resize to match ViT/ResNet expected input"""
        if tensor.shape[-2:] != size:
            tensor = F.interpolate(
                tensor.unsqueeze(0), size=size, mode='bilinear', align_corners=False
            ).squeeze(0)
        return tensor

    if use_tta:
        tta_transforms = AutoAugmentTransforms.get_tta_transforms()
        predictions = []

        for transform in tta_transforms:
            augmented = transform(image=image)['image']
            augmented = resize_if_needed(augmented)
            augmented = augmented.unsqueeze(0).to(device)

            with torch.no_grad():
                output = model(augmented)
                prob = F.softmax(output, dim=1)
                predictions.append(prob.cpu().numpy())

        avg_prob = np.mean(predictions, axis=0)[0]
    else:
        transform = AutoAugmentTransforms.get_valid_transforms()
        augmented = transform(image=image)['image']
        augmented = resize_if_needed(augmented)
        augmented = augmented.unsqueeze(0).to(device)

        with torch.no_grad():
            output = model(augmented)
            avg_prob = F.softmax(output, dim=1).cpu().numpy()[0]

    pred_class = np.argmax(avg_prob)
    confidence = avg_prob[pred_class]

    result = {
        'prediction': class_names[pred_class],
        'confidence': confidence,
        'probabilities': {
            'Non-Autistic': avg_prob[0],
            'Autistic': avg_prob[1]
        }
    }

    return result


# Example: Test inference on a random image
sample_test_image = os.path.join(config.TEST_DIR, os.listdir(config.TEST_DIR)[0])
result = predict_single_image(sample_test_image, ensemble.models[0], use_tta=True)

print("\n" + "="*70)
print("SAMPLE INFERENCE")
print("="*70)
print(f"Image: {sample_test_image}")
print(f"Prediction: {result['prediction']}")
print(f"Confidence: {result['confidence']:.2%}")
print(f"Probabilities:")
print(f"  Non-Autistic: {result['probabilities']['Non-Autistic']:.4f}")
print(f"  Autistic: {result['probabilities']['Autistic']:.4f}")